# Lesson 4 — Statistics Refresher

Every surrogate model in this project — Gaussian Process, MC Dropout, Deep Ensembles — outputs exactly two numbers for each prediction:

- **Mean (μ)** — the best guess
- **Variance (σ²)** — how uncertain the surrogate is

Together they define a **Gaussian distribution**. This lesson explains what that means.

By the end you will understand:
1. What a Gaussian distribution is and how to read it
2. What covariance measures
3. How a 2D Gaussian works
4. What a conditional distribution is — the exact operation a GP performs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import norm, multivariate_normal

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

print('Imports OK')

---
## Part 1 — The 1D Gaussian Distribution

A Gaussian (normal) distribution is fully described by two numbers:
- **μ (mu)** — the mean: where the peak is
- **σ (sigma)** — the standard deviation: how wide the bell is

The probability density function (PDF) is:
$$p(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

You do not need to memorise this formula. What matters: **narrow = confident, wide = uncertain**.

In [ ]:
x = np.linspace(-6, 6, 400)

configs = [
    (0.0, 0.5, 'steelblue', 'μ=0, σ=0.5  (narrow = confident)'),
    (0.0, 1.0, 'tomato',    'μ=0, σ=1.0  (standard normal)'),
    (0.0, 2.0, 'green',     'μ=0, σ=2.0  (wide = uncertain)'),
    (2.0, 1.0, 'purple',    'μ=2, σ=1.0  (shifted right)'),
]

fig, ax = plt.subplots(figsize=(9, 5))
for mu, sigma, color, label in configs:
    ax.plot(x, norm.pdf(x, mu, sigma), color=color, linewidth=2.5, label=label)

ax.set_title('Gaussian PDFs with different μ and σ')
ax.set_xlabel('x')
ax.set_ylabel('Probability density p(x)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Surrogate interpretation:**
- When a GP or NN says μ=1.5, σ=0.1 → it is very confident the answer is near 1.5
- When it says μ=1.5, σ=2.0 → it thinks the answer is around 1.5 but is very uncertain

Wide uncertainty bands → the surrogate hasn't seen nearby data → explore that region!

### The 68-95-99.7 Rule

For any Gaussian:
- **68%** of values fall within ±1σ of the mean
- **95%** fall within ±2σ
- **99.7%** fall within ±3σ

When a surrogate shows a ±2σ band, it means the true value falls inside the band 95% of the time.

In [ ]:
mu, sigma = 0.0, 1.0
pdf = norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, pdf, color='steelblue', linewidth=2.5)

bands = [(1, 0.5, '±1σ: 68.3%'), (2, 0.3, '±2σ: 95.4%'), (3, 0.15, '±3σ: 99.7%')]
for n_std, alpha, label in bands:
    mask = np.abs(x - mu) <= n_std * sigma
    ax.fill_between(x, pdf, where=mask, alpha=alpha, color='steelblue', label=label)

ax.set_title('68-95-99.7 Rule (standard normal μ=0, σ=1)')
ax.set_xlabel('x')
ax.set_ylabel('Probability density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Try it:** Change `sigma = 1.0` to `sigma = 0.3` and rerun. What does a very narrow Gaussian mean for a surrogate?

---
## Part 2 — Sampling from a Gaussian

Drawing a **sample** from N(μ, σ²) means picking a random value according to the bell curve probability.

MC Dropout (Lesson 10) works by drawing multiple samples from the neural network's output distribution and computing their mean and variance. So sampling is not just theory — it is the core operation of the algorithm.

In [ ]:
true_mu, true_sigma = 1.5, 0.8

# Draw samples and see that mean + variance converge to truth
for n in [5, 20, 100, 1000]:
    samples = rng.normal(true_mu, true_sigma, n)
    print(f'n={n:5d}:  sample mean={samples.mean():.3f}  (true={true_mu})  '
          f'sample std={samples.std():.3f}  (true={true_sigma})')

In [ ]:
# Running mean converges to true μ as n grows
n_max = 500
samples_all = rng.normal(true_mu, true_sigma, n_max)
running_mean = np.cumsum(samples_all) / np.arange(1, n_max + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(running_mean, color='steelblue', linewidth=1.5, label='Running sample mean')
ax.axhline(true_mu, color='tomato', linestyle='--', linewidth=2, label=f'True μ = {true_mu}')
ax.fill_between(range(n_max),
                true_mu - true_sigma, true_mu + true_sigma,
                alpha=0.15, color='tomato', label='±1σ band')
ax.set_title('Sample mean converges to true μ (Law of Large Numbers)')
ax.set_xlabel('Number of samples')
ax.set_ylabel('Running mean')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**MC Dropout connection:** In Lesson 10, we will run 50 forward passes through a network with dropout ON. Each pass gives a slightly different output — 50 samples from the network's implicit distribution. The mean of 50 samples = our prediction. The variance of 50 samples = our uncertainty.

**Try it:** Change `n_max = 500` to `n_max = 10`. How stable is the running mean with just 10 samples?

---
## Part 3 — Covariance and Correlation

**Covariance** measures how two variables move together:
- Positive covariance: when X goes up, Y tends to go up
- Negative covariance: when X goes up, Y tends to go down
- Zero covariance: X and Y move independently

**Correlation** is normalised covariance — always between −1 and +1:
$$\rho = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}$$

In [ ]:
def plot_confidence_ellipse(ax, cov, color):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    for n_std, alpha in [(1, 0.7), (2, 0.4)]:
        w, h = 2 * n_std * np.sqrt(vals)
        ell = Ellipse((0, 0), width=w, height=h, angle=angle,
                      edgecolor=color, facecolor='none', linewidth=2.5, alpha=alpha)
        ax.add_patch(ell)

cov_configs = [
    (np.array([[1.0,  0.8], [ 0.8, 1.0]]), 'Positive correlation ρ=+0.8', 'steelblue'),
    (np.array([[1.0,  0.0], [ 0.0, 1.0]]), 'No correlation ρ=0.0',        'green'),
    (np.array([[1.0, -0.8], [-0.8, 1.0]]), 'Negative correlation ρ=−0.8', 'tomato'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (cov, title, color) in zip(axes, cov_configs):
    samples = rng.multivariate_normal([0, 0], cov, size=300)
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.4, s=20, color=color)
    plot_confidence_ellipse(ax, cov, color)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_title(title)
    ax.set_xlabel('X₁')
    ax.set_ylabel('X₂')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Covariance shapes — 1σ and 2σ ellipses', fontsize=12)
plt.tight_layout()
plt.show()

**GP connection:** A Gaussian Process uses a covariance (kernel) function to say: "nearby input points should have similar output values" — which is exactly positive correlation in input space. That is Lesson 5.

**Try it:** Change `0.8` to `0.99` in the first covariance matrix. What shape do the points form? What does perfect correlation mean?

---
## Part 4 — The 2D Multivariate Gaussian

A multivariate Gaussian over d variables is defined by:
- A **mean vector** μ ∈ ℝᵈ
- A **covariance matrix** Σ ∈ ℝᵈˣᵈ — positive semi-definite

A Gaussian Process is essentially an infinite-dimensional multivariate Gaussian — one dimension per input point. The covariance matrix is computed from the kernel function.

In [ ]:
mu_2d  = np.array([1.0, -0.5])
cov_2d = np.array([[1.2, 0.7],
                   [0.7, 0.8]])

print('Mean vector μ:')
print(mu_2d)
print('\nCovariance matrix Σ:')
print(cov_2d)
print(f'\nVar(X₁) = {cov_2d[0,0]}  →  std(X₁) = {np.sqrt(cov_2d[0,0]):.3f}')
print(f'Var(X₂) = {cov_2d[1,1]}  →  std(X₂) = {np.sqrt(cov_2d[1,1]):.3f}')
print(f'Cov(X₁,X₂) = {cov_2d[0,1]}')
print(f'Correlation ρ = {cov_2d[0,1] / (np.sqrt(cov_2d[0,0]) * np.sqrt(cov_2d[1,1])):.3f}')

In [ ]:
grid_x = np.linspace(-3, 5, 200)
grid_y = np.linspace(-4, 3, 200)
X2, Y2 = np.meshgrid(grid_x, grid_y)
Z2 = multivariate_normal.pdf(np.dstack((X2, Y2)), mean=mu_2d, cov=cov_2d)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cf = axes[0].contourf(X2, Y2, Z2, levels=15, cmap='viridis')
axes[0].contour(X2, Y2, Z2, levels=15, colors='white', linewidths=0.4, alpha=0.5)
fig.colorbar(cf, ax=axes[0], label='Density')
axes[0].plot(*mu_2d, 'r*', markersize=14, label=f'mean {mu_2d.tolist()}')
axes[0].set_title('2D Gaussian — contour')
axes[0].set_xlabel('X₁')
axes[0].set_ylabel('X₂')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

samples_2d = rng.multivariate_normal(mu_2d, cov_2d, size=400)
axes[1].scatter(samples_2d[:, 0], samples_2d[:, 1], alpha=0.4, s=20, color='steelblue',
                label='400 samples')
axes[1].plot(*mu_2d, 'r*', markersize=14, label='true mean')
sm = samples_2d.mean(axis=0)
axes[1].plot(*sm, 'g^', markersize=10, label=f'sample mean ({sm[0]:.2f}, {sm[1]:.2f})')
axes[1].set_title('2D Gaussian — samples')
axes[1].set_xlabel('X₁')
axes[1].set_ylabel('X₂')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Sample covariance matrix (should be close to true Σ):')
print(np.cov(samples_2d.T).round(3))

**Try it:** Change `cov_2d` so that the off-diagonal is `0.0`. What shape does the contour take?

---
## Part 5 — Conditional Distributions

This is the most important concept in this lesson.

**Conditional distribution:** Given that X takes a specific value, what does Y look like?

For a 2D Gaussian with correlation ρ:
$$Y \mid X = x \sim \mathcal{N}(\rho x,\ 1 - \rho^2)$$

- **Conditional mean:** ρ·x — shifts based on what we observed
- **Conditional variance:** 1 − ρ² — always smaller than marginal variance (observing X reduces uncertainty about Y)

This is **exactly** what a Gaussian Process computes at every new point.

In [ ]:
rho = 0.85
cov_xy = np.array([[1.0, rho], [rho, 1.0]])

print(f'Correlation ρ = {rho}')
print(f'Conditional variance = 1 - ρ² = {1 - rho**2:.3f}')
print(f'Conditional std       = √(1-ρ²) = {np.sqrt(1 - rho**2):.3f}')
print()
for xv in [-1.5, 0.0, 1.5]:
    cond_mu = rho * xv
    print(f'  X={xv:5.1f}  →  E[Y|X] = {cond_mu:.3f},  std[Y|X] = {np.sqrt(1-rho**2):.3f}')

In [ ]:
x_vals   = [-1.5, 0.0, 1.5]
colors_c = ['tomato', 'steelblue', 'green']
y_range  = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Joint scatter
joint_samples = rng.multivariate_normal([0, 0], cov_xy, size=500)
axes[0].scatter(joint_samples[:, 0], joint_samples[:, 1],
                alpha=0.3, s=15, color='gray', label='joint samples')
for xv, color in zip(x_vals, colors_c):
    axes[0].axvline(xv, color=color, linewidth=2, linestyle='--', label=f'X = {xv}')
    axes[0].plot(xv, rho * xv, 'o', color=color, markersize=10)

axes[0].set_title(f'Joint distribution — conditioning on X = x  (ρ={rho})')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Conditional PDFs
cond_sigma = np.sqrt(1 - rho**2)
for xv, color in zip(x_vals, colors_c):
    cond_mu = rho * xv
    axes[1].plot(y_range, norm.pdf(y_range, cond_mu, cond_sigma), color=color, linewidth=2,
                 label=f'P(Y|X={xv}):  μ={cond_mu:.2f}')
    axes[1].axvline(cond_mu, color=color, linestyle=':', linewidth=1.2)

axes[1].set_title('Conditional distributions P(Y | X = x)')
axes[1].set_xlabel('y')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Conditional Gaussian — observing X shifts the mean, reduces variance', fontsize=11)
plt.tight_layout()
plt.show()

Key observations:
- All three conditional distributions have the **same width** (same variance 1−ρ²)
- But their **means shift** based on the observed X value
- Observing X gives us **less uncertainty** about Y than before (conditional var < marginal var)

**Try it:** Change `rho = 0.85` to `rho = 0.3`. How does the conditional distribution change? What does low correlation mean for uncertainty reduction?

---
## Part 6 — Connecting to GP Predictions

A Gaussian Process makes predictions by computing exactly the conditional distribution from Part 5 — but in function space.

Given observations (x_train, y_train), the GP asks:
> "What is the distribution of f(x*) given what I have seen?"

The answer is a Gaussian: **P(f(x*) | data) = N(μ*, σ*²)**

- **μ*** = conditional mean = best prediction at x*
- **σ*²** = conditional variance = uncertainty at x* (high where data is sparse)

In [ ]:
# Minimal GP with RBF kernel — just for illustration
def rbf_kernel(a, b, length=0.2, var=1.0):
    return var * np.exp(-0.5 * ((a[:, None] - b[None, :]) / length)**2)

x_obs = np.array([0.1, 0.4, 0.7, 0.9])
y_obs = np.sin(2 * np.pi * x_obs)
x_star = np.linspace(0, 1, 200)

K   = rbf_kernel(x_obs, x_obs) + 1e-6 * np.eye(len(x_obs))
Ks  = rbf_kernel(x_star, x_obs)
Kss = rbf_kernel(x_star, x_star)
L   = np.linalg.cholesky(K)

mu_gp  = Ks @ np.linalg.solve(L.T, np.linalg.solve(L, y_obs))
v      = np.linalg.solve(L, Ks.T)
std_gp = np.sqrt(np.maximum(np.diag(Kss) - np.sum(v**2, axis=0), 0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(x_star, np.sin(2*np.pi*x_star), 'gray', linewidth=1.5, linestyle='--',
             label='True f(x)', alpha=0.6)
axes[0].plot(x_star, mu_gp, 'steelblue', linewidth=2, label='GP mean μ*')
axes[0].fill_between(x_star, mu_gp - 2*std_gp, mu_gp + 2*std_gp,
                     alpha=0.25, color='steelblue', label='±2σ* (95% credible)')
axes[0].scatter(x_obs, y_obs, color='black', s=80, zorder=5, label='Observed data')
axes[0].set_title('GP prediction = conditional Gaussian at each x*')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Predictive Gaussian at two query points
x_queries = [0.3, 0.6]
colors_q  = ['tomato', 'green']
y_range2  = np.linspace(-2.5, 2.5, 300)

for xq, color in zip(x_queries, colors_q):
    idx   = np.argmin(np.abs(x_star - xq))
    mu_q  = mu_gp[idx]
    std_q = std_gp[idx]
    axes[1].plot(y_range2, norm.pdf(y_range2, mu_q, std_q), color=color, linewidth=2.5,
                 label=f'x*={xq}: μ={mu_q:.2f}, σ={std_q:.2f}')
    axes[1].axvline(mu_q, color=color, linestyle=':', linewidth=1.2)

axes[1].set_title('GP output: a Gaussian at every prediction point')
axes[1].set_xlabel('f(x*)')
axes[1].set_ylabel('Density p(f(x*) | data)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('GP prediction = conditional Gaussian — exactly what Part 5 computed',
             fontsize=11)
plt.tight_layout()
plt.show()

Notice:
- At x*=0.3 (between observations): the distribution is narrow → GP is fairly confident
- At x*=0.6 (further from data): the distribution is wider → more uncertain

The acquisition functions in Lesson 7 use exactly these μ and σ values to decide where to evaluate next. High σ → high uncertainty → explore.

---
## Summary

| Concept | What it means for surrogates |
|---|---|
| Gaussian distribution | Output of every surrogate: a bell curve defined by μ and σ² |
| Mean μ | Best prediction at that point |
| Variance σ² | Uncertainty — wide = surrogate has not seen nearby data |
| Covariance | How similar two outputs are — forms the GP kernel matrix |
| Conditional distribution | The exact computation a GP does to make a prediction |
| 68-95-99.7 rule | ±2σ band covers 95% of the true distribution |

---

## Exercises

1. In Part 1, change `sigma=2.0` to `sigma=0.1`. What does a very narrow surrogate uncertainty band mean?
2. In Part 3, change the covariance to `[[1, -0.9], [-0.9, 1]]`. What shape do the samples form?
3. In Part 5, set `rho = 0.1`. How does the conditional distribution change when variables are nearly uncorrelated?

---

## What's Next

**Lesson 5** — Kernels and similarity functions. You will use the covariance intuition from this lesson to build the RBF kernel — the engine of the Gaussian Process surrogate.